In [ ]:
# ------------------------------------------------------------
# 05 Dashboard Design Decisions — setup cell
# Loads calls data, MCPP geography, response-time data, population,
# and shared constants needed by dashboard figure cells.
# ------------------------------------------------------------

from pathlib import Path
import sys
import json
import itertools
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from dashboard.spd_snapshot import load_spd_call_snapshot

# ------------------------------------------------------------
# Shared constants
# ------------------------------------------------------------

EVENT_ID_COLUMN = "cad_event_number"
ROW_ID_COLUMN = "call_sign_dispatch_id"
TIME_COLUMN = "cad_event_original_time_queued"
ARRIVAL_TIME_COLUMN = "cad_event_arrived_time"

LAT_COL = "dispatch_latitude"
LON_COL = "dispatch_longitude"

PLOTLY_TEMPLATE = "plotly_dark"
PLOT_BG = "#545455"
PAPER_BG = "#111111"

PLOTLY_SEATTLE_CENTER = {
    "lat": 47.6062,
    "lon": -122.3321,
}

PLOTLY_MAP_STYLE = "carto-darkmatter"

SEATTLE_CENTER = [47.6062, -122.3321]

IMPORTANT_EVENT_GROUPS = [
    "assault",
    "burglary",
    "domestic disturbance/violence",
    "kidnap",
    "rape",
    "robbery",
    "sex offenses (non-rape)",
    "theft",
    "narcotics",
    "homicide",
]

pio.renderers.default = "notebook"

# ------------------------------------------------------------
# Load SPD calls snapshot for daily volume and map points
# ------------------------------------------------------------

df, metadata = load_spd_call_snapshot(
    PROJECT_ROOT / "data" / "processed"
)

df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN], errors="coerce")
df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")
df[LON_COL] = pd.to_numeric(df[LON_COL], errors="coerce")

for col in [
    "event_group",
    "dispatch_neighborhood",
    "dispatch_precinct",
    "dispatch_sector",
    "dispatch_beat",
    "priority",
    "initial_call_type",
    "final_call_type",
]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.lower()
        )

df["date"] = df[TIME_COLUMN].dt.date

valid_time = df[
    df[TIME_COLUMN].notna()
    & df[EVENT_ID_COLUMN].notna()
].copy()

# ------------------------------------------------------------
# Load official MCPP boundaries
# ------------------------------------------------------------

GEO_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "geography"
GEO_EXTERNAL_DIR = PROJECT_ROOT / "data" / "external" / "boundaries"

processed_mcpp_geojson_path = GEO_PROCESSED_DIR / "spd_mcpp_boundaries.geojson"
external_mcpp_geojson_path = GEO_EXTERNAL_DIR / "spd_mcpp_boundaries.geojson"

MCPP_GEOJSON_URL = (
    "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/ArcGIS/rest/services/"
    "SPD_Boundaries/FeatureServer/0/query"
    "?where=1%3D1"
    "&outFields=*"
    "&outSR=4326"
    "&f=geojson"
)

if processed_mcpp_geojson_path.exists():
    mcpp_boundaries = gpd.read_file(processed_mcpp_geojson_path)
elif external_mcpp_geojson_path.exists():
    mcpp_boundaries = gpd.read_file(external_mcpp_geojson_path)
else:
    GEO_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    GEO_EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

    mcpp_boundaries = gpd.read_file(MCPP_GEOJSON_URL)
    mcpp_boundaries.to_file(processed_mcpp_geojson_path, driver="GeoJSON")
    mcpp_boundaries.to_file(external_mcpp_geojson_path, driver="GeoJSON")

if mcpp_boundaries.crs is not None:
    mcpp_boundaries = mcpp_boundaries.to_crs(epsg=4326)
else:
    mcpp_boundaries = mcpp_boundaries.set_crs(epsg=4326)

mcpp_boundaries.columns = [
    col.lower().strip()
    for col in mcpp_boundaries.columns
]

if "mcpp_neighborhood" not in mcpp_boundaries.columns:
    mcpp_boundaries["mcpp_neighborhood"] = (
        mcpp_boundaries["neighborhood"]
        .astype("string")
        .str.strip()
        .str.lower()
    )
else:
    mcpp_boundaries["mcpp_neighborhood"] = (
        mcpp_boundaries["mcpp_neighborhood"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

if "mcpp_precinct" not in mcpp_boundaries.columns:
    mcpp_boundaries["mcpp_precinct"] = (
        mcpp_boundaries["precinct"]
        .astype("string")
        .str.strip()
        .str.lower()
    )
else:
    mcpp_boundaries["mcpp_precinct"] = (
        mcpp_boundaries["mcpp_precinct"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

if "objectid" not in mcpp_boundaries.columns:
    mcpp_boundaries["objectid"] = range(1, len(mcpp_boundaries) + 1)

mcpp_boundaries = mcpp_boundaries[
    [
        "objectid",
        "mcpp_neighborhood",
        "mcpp_precinct",
        "geometry",
    ]
].copy()

mcpp_boundaries["plot_feature_id"] = (
    mcpp_boundaries["objectid"]
    .astype(str)
)

# ------------------------------------------------------------
# Build or load event-to-MCPP lookup
# ------------------------------------------------------------

mcpp_event_lookup_path = GEO_PROCESSED_DIR / "event_mcpp_lookup.parquet"

mappable_events = (
    df[
        df[EVENT_ID_COLUMN].notna()
        & df[TIME_COLUMN].notna()
        & df[LAT_COL].notna()
        & df[LON_COL].notna()
        & df[LAT_COL].between(47.45, 47.75)
        & df[LON_COL].between(-122.46, -122.20)
    ]
    .sort_values(TIME_COLUMN, ascending=False)
    .drop_duplicates(subset=EVENT_ID_COLUMN)
    .copy()
)

if mcpp_event_lookup_path.exists():
    event_mcpp_lookup = pd.read_parquet(mcpp_event_lookup_path)
else:
    event_points_gdf = gpd.GeoDataFrame(
        mappable_events,
        geometry=gpd.points_from_xy(
            mappable_events[LON_COL],
            mappable_events[LAT_COL],
        ),
        crs="EPSG:4326",
    )

    event_mcpp_lookup_gdf = gpd.sjoin(
        event_points_gdf[[EVENT_ID_COLUMN, "geometry"]],
        mcpp_boundaries[
            [
                "mcpp_neighborhood",
                "mcpp_precinct",
                "geometry",
            ]
        ],
        how="left",
        predicate="within",
    ).drop(columns=["index_right"], errors="ignore")

    event_mcpp_lookup = (
        event_mcpp_lookup_gdf[
            [
                EVENT_ID_COLUMN,
                "mcpp_neighborhood",
                "mcpp_precinct",
            ]
        ]
        .dropna(subset=[EVENT_ID_COLUMN])
        .drop_duplicates(subset=EVENT_ID_COLUMN)
        .copy()
    )

    GEO_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    event_mcpp_lookup.to_parquet(mcpp_event_lookup_path, index=False)

event_mcpp_lookup["mcpp_neighborhood"] = (
    event_mcpp_lookup["mcpp_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

event_mcpp_lookup["mcpp_precinct"] = (
    event_mcpp_lookup["mcpp_precinct"]
    .astype("string")
    .str.strip()
    .str.lower()
)

event_mcpp = mappable_events.merge(
    event_mcpp_lookup,
    on=EVENT_ID_COLUMN,
    how="left",
)

event_mcpp[TIME_COLUMN] = pd.to_datetime(
    event_mcpp[TIME_COLUMN],
    errors="coerce"
)

latest_event_time = event_mcpp[TIME_COLUMN].max()

# ------------------------------------------------------------
# Build important_share_choropleth for dashboard map
# ------------------------------------------------------------

mcpp_choropleth_gdf = mcpp_boundaries.copy()

total_mcpp_counts = (
    event_mcpp
    .dropna(subset=["mcpp_neighborhood"])
    .groupby("mcpp_neighborhood", as_index=False)
    .agg(
        total_event_count=(EVENT_ID_COLUMN, "nunique"),
    )
)

important_mcpp_counts = (
    event_mcpp[
        event_mcpp["event_group"].isin(IMPORTANT_EVENT_GROUPS)
    ]
    .dropna(subset=["mcpp_neighborhood"])
    .groupby("mcpp_neighborhood", as_index=False)
    .agg(
        important_event_count=(EVENT_ID_COLUMN, "nunique"),
    )
)

important_share_counts = total_mcpp_counts.merge(
    important_mcpp_counts,
    on="mcpp_neighborhood",
    how="left",
)

important_share_counts["important_event_count"] = (
    important_share_counts["important_event_count"]
    .fillna(0)
    .astype(int)
)

important_share_counts["important_event_share"] = (
    important_share_counts["important_event_count"]
    / important_share_counts["total_event_count"]
    * 100
)

important_share_choropleth = mcpp_choropleth_gdf.merge(
    important_share_counts,
    on="mcpp_neighborhood",
    how="left",
)

important_share_choropleth["total_event_count"] = (
    important_share_choropleth["total_event_count"]
    .fillna(0)
    .astype(int)
)

important_share_choropleth["important_event_count"] = (
    important_share_choropleth["important_event_count"]
    .fillna(0)
    .astype(int)
)

important_share_choropleth["important_event_share"] = (
    important_share_choropleth["important_event_share"]
    .fillna(0)
)

# ------------------------------------------------------------
# Load response-time analysis data
# ------------------------------------------------------------

RESPONSE_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "response_time"
full_history_response_path = (
    RESPONSE_OUTPUT_DIR / "spd_event_response_times_2009_2026.parquet"
)

if full_history_response_path.exists():
    response_analysis = pd.read_parquet(full_history_response_path)

    response_analysis["queued_time"] = pd.to_datetime(
        response_analysis["queued_time"],
        errors="coerce",
    )

else:
    response_df = df.copy()

    response_df[ARRIVAL_TIME_COLUMN] = pd.to_datetime(
        response_df[ARRIVAL_TIME_COLUMN],
        errors="coerce",
    )

    event_response = (
        response_df
        .dropna(subset=[EVENT_ID_COLUMN])
        .sort_values(TIME_COLUMN)
        .groupby(EVENT_ID_COLUMN, as_index=False)
        .agg(
            queued_time=(TIME_COLUMN, "min"),
            first_arrival_time=(ARRIVAL_TIME_COLUMN, "min"),
            event_group=("event_group", "first"),
            priority=("priority", "first"),
            dispatch_neighborhood=("dispatch_neighborhood", "first"),
            dispatch_records=(ROW_ID_COLUMN, "nunique"),
        )
    )

    event_response["response_time_minutes"] = (
        event_response["first_arrival_time"] - event_response["queued_time"]
    ).dt.total_seconds() / 60

    response_analysis = event_response.copy()

response_analysis = response_analysis[
    response_analysis["response_time_minutes"].notna()
    & (response_analysis["response_time_minutes"] >= 0)
    & (response_analysis["response_time_minutes"] <= 24 * 60)
].copy()

if "queued_time" in response_analysis.columns:
    response_analysis["queued_time"] = pd.to_datetime(
        response_analysis["queued_time"],
        errors="coerce",
    )

if "dispatch_neighborhood" in response_analysis.columns:
    response_analysis["dispatch_neighborhood"] = (
        response_analysis["dispatch_neighborhood"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

if "event_group" in response_analysis.columns:
    response_analysis["event_group"] = (
        response_analysis["event_group"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# ------------------------------------------------------------
# Event importance bins
# ------------------------------------------------------------

violent_or_person_crime_groups = [
    "assault",
    "domestic disturbance/violence",
    "homicide",
    "kidnap",
    "rape",
    "robbery",
    "sex offenses (non-rape)",
]

drug_related_groups = [
    "narcotics",
]

property_or_nonviolent_groups = [
    "burglary",
    "theft",
    "car prowl",
    "fraud",
    "trespass",
]

lower_public_safety_groups = [
    "traffic",
    "disturbance",
    "misc. misdemeanors & violations",
    "warrant services/order",
    "person down/injury",
    "suspicious circumstances",
]


def assign_event_importance_bin(event_group):
    if pd.isna(event_group):
        return "unknown / unclassified"

    event_group = str(event_group).strip().lower()

    if event_group in violent_or_person_crime_groups:
        return "violent/person crime"

    if event_group in drug_related_groups:
        return "drug-related"

    if event_group in property_or_nonviolent_groups:
        return "property/nonviolent"

    if event_group in lower_public_safety_groups:
        return "lower public-safety urgency"

    return "other / unclassified"


response_analysis["event_importance_bin"] = (
    response_analysis["event_group"]
    .apply(assign_event_importance_bin)
)

# ------------------------------------------------------------
# Load neighborhood population for per-capita scatterplot
# ------------------------------------------------------------

population_path = PROJECT_ROOT / "data" / "external" / "neighborhood_population.csv"

neighborhood_population = pd.read_csv(population_path)

neighborhood_population["dispatch_neighborhood"] = (
    neighborhood_population["dispatch_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

neighborhood_population["population"] = pd.to_numeric(
    neighborhood_population["population"],
    errors="coerce",
)

years_observed = (
    response_analysis["queued_time"].max()
    - response_analysis["queued_time"].min()
).days / 365.25

# ------------------------------------------------------------
# Setup check
# ------------------------------------------------------------

print("Setup complete.")
print(f"Snapshot rows: {len(df):,}")
print(f"Valid-time rows: {len(valid_time):,}")
print(f"Mappable events: {mappable_events[EVENT_ID_COLUMN].nunique():,}")
print(f"MCPP-matched map events: {event_mcpp['mcpp_neighborhood'].notna().sum():,}")
print(f"Response-analysis events: {response_analysis[EVENT_ID_COLUMN].nunique():,}")
print(f"Years observed for response analysis: {years_observed:.2f}")
print(f"MCPP boundary polygons: {len(mcpp_boundaries):,}")

In [ ]:
import itertools

TARGET_IMPORTANCE_BINS = [
    "property/nonviolent",
    "drug-related",
    "violent/person crime",
]

def make_bin_combo_label(bin_combo):
    if len(bin_combo) == len(TARGET_IMPORTANCE_BINS):
        return "All selected bins"
    return " + ".join(bin_combo)


bin_combinations = []

for r in range(1, len(TARGET_IMPORTANCE_BINS) + 1):
    for combo in itertools.combinations(TARGET_IMPORTANCE_BINS, r):
        bin_combinations.append(list(combo))

bin_combinations = (
    [TARGET_IMPORTANCE_BINS]
    + [
        combo for combo in bin_combinations
        if combo != TARGET_IMPORTANCE_BINS
    ]
)


def ensure_event_importance_bin(data):
    out = data.copy()

    if "event_importance_bin" not in out.columns:
        out["event_group"] = (
            out["event_group"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        out["event_importance_bin"] = (
            out["event_group"]
            .apply(assign_event_importance_bin)
        )

    return out

In [ ]:
# ------------------------------------------------------------
# Fig 1 - Map
# Choropleth color:
#   Past-year unique CAD events per 1,000 residents
#   for selected event importance bin(s)
#
# Point layer:
#   Latest available day events in selected importance bin(s)
#
# Dropdown:
#   Event importance bin combinations
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import json

# ------------------------------------------------------------
# Prep event data
# ------------------------------------------------------------

event_mcpp_bins = ensure_event_importance_bin(event_mcpp)

event_mcpp_bins[TIME_COLUMN] = pd.to_datetime(
    event_mcpp_bins[TIME_COLUMN],
    errors="coerce",
)

event_mcpp_bins[LAT_COL] = pd.to_numeric(
    event_mcpp_bins[LAT_COL],
    errors="coerce",
)

event_mcpp_bins[LON_COL] = pd.to_numeric(
    event_mcpp_bins[LON_COL],
    errors="coerce",
)

for col in [
    "event_group",
    "event_importance_bin",
    "mcpp_neighborhood",
    "mcpp_precinct",
    "priority",
    "initial_call_type",
    "final_call_type",
]:
    if col not in event_mcpp_bins.columns:
        event_mcpp_bins[col] = pd.NA

    event_mcpp_bins[col] = (
        event_mcpp_bins[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

event_mcpp_bins = event_mcpp_bins[
    event_mcpp_bins[TIME_COLUMN].notna()
    & event_mcpp_bins[EVENT_ID_COLUMN].notna()
    & event_mcpp_bins[LAT_COL].notna()
    & event_mcpp_bins[LON_COL].notna()
].copy()

latest_available_day = event_mcpp_bins[TIME_COLUMN].dt.normalize().max()

past_year_start = latest_available_day - pd.Timedelta(days=364)

# Choropleth and available point data both use the past year ending
# at the latest available dataset date.
past_year_events = event_mcpp_bins[
    event_mcpp_bins[TIME_COLUMN].dt.normalize().between(
        past_year_start,
        latest_available_day,
    )
].copy()

# Important:
# point_events intentionally contains the full past-year point universe.
# The browser dashboard will filter these map points based on Fig 2's
# visible x-axis/rangeslider window.
point_events = past_year_events.copy()

point_events["event_date_for_filter"] = (
    point_events[TIME_COLUMN]
    .dt.normalize()
    .dt.strftime("%Y-%m-%d")
)

point_events["event_time_display"] = (
    point_events[TIME_COLUMN]
    .dt.strftime("%Y-%m-%d %H:%M")
)

print(f"Map choropleth window: {past_year_start.date()} to {latest_available_day.date()}")
print(f"Map point data available for filtering: {past_year_start.date()} to {latest_available_day.date()}")

# ------------------------------------------------------------
# Compute response time directly from rolling snapshot if possible
# ------------------------------------------------------------

if "response_time_minutes" not in past_year_events.columns:
    if ARRIVAL_TIME_COLUMN in past_year_events.columns:
        past_year_events[ARRIVAL_TIME_COLUMN] = pd.to_datetime(
            past_year_events[ARRIVAL_TIME_COLUMN],
            errors="coerce",
        )

        past_year_events["response_time_minutes"] = (
            past_year_events[ARRIVAL_TIME_COLUMN] - past_year_events[TIME_COLUMN]
        ).dt.total_seconds() / 60

    else:
        past_year_events["response_time_minutes"] = np.nan

past_year_response_events = past_year_events[
    past_year_events["response_time_minutes"].notna()
    & (past_year_events["response_time_minutes"] >= 0)
    & (past_year_events["response_time_minutes"] <= 24 * 60)
].copy()

# ------------------------------------------------------------
# Population matched to MCPP neighborhood
# ------------------------------------------------------------

population_for_mcpp = neighborhood_population.copy()

if "mcpp_neighborhood" not in population_for_mcpp.columns:
    population_for_mcpp = population_for_mcpp.rename(
        columns={"dispatch_neighborhood": "mcpp_neighborhood"}
    )

population_for_mcpp["mcpp_neighborhood"] = (
    population_for_mcpp["mcpp_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

population_for_mcpp["population"] = pd.to_numeric(
    population_for_mcpp["population"],
    errors="coerce",
)

# ------------------------------------------------------------
# Base MCPP boundary layer
# ------------------------------------------------------------

base_gdf = mcpp_choropleth_gdf.copy()

base_gdf["plot_feature_id"] = (
    base_gdf["objectid"]
    .astype(str)
)

base_gdf["mcpp_neighborhood"] = (
    base_gdf["mcpp_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

base_gdf["mcpp_neighborhood_display"] = (
    base_gdf["mcpp_neighborhood"]
    .astype("string")
    .str.title()
)

base_gdf = base_gdf.drop(
    columns=["population"],
    errors="ignore",
)

base_gdf = base_gdf.merge(
    population_for_mcpp[
        [
            "mcpp_neighborhood",
            "population",
        ]
    ],
    on="mcpp_neighborhood",
    how="left",
)

base_gdf["population"] = pd.to_numeric(
    base_gdf["population"],
    errors="coerce",
)

base_geojson = json.loads(base_gdf.to_json())

# ------------------------------------------------------------
# Build dropdown map
# ------------------------------------------------------------

fig_map = go.Figure()

trace_metadata = []

for combo_i, bin_combo in enumerate(bin_combinations):
    combo_label = make_bin_combo_label(bin_combo)
    combo_visible = combo_i == 0

    # --------------------------------------------------------
    # Choropleth metric:
    # Past-year unique CAD events per 1,000 residents
    # in selected importance bin(s)
    # --------------------------------------------------------

    combo_events = past_year_events[
        past_year_events["event_importance_bin"].isin(bin_combo)
    ].copy()

    combo_counts = (
        combo_events
        .dropna(subset=["mcpp_neighborhood"])
        .groupby("mcpp_neighborhood", as_index=False)
        .agg(
            past_year_unique_events=(EVENT_ID_COLUMN, "nunique"),
        )
    )

    combo_response_events = past_year_response_events[
        past_year_response_events["event_importance_bin"].isin(bin_combo)
    ].copy()

    combo_response_summary = (
        combo_response_events
        .dropna(subset=["mcpp_neighborhood", "response_time_minutes"])
        .groupby("mcpp_neighborhood", as_index=False)
        .agg(
            selected_median_response_minutes=("response_time_minutes", "median"),
        )
    )

    combo_gdf = (
        base_gdf
        .merge(
            combo_counts,
            on="mcpp_neighborhood",
            how="left",
        )
        .merge(
            combo_response_summary,
            on="mcpp_neighborhood",
            how="left",
        )
    )

    combo_gdf["past_year_unique_events"] = (
        combo_gdf["past_year_unique_events"]
        .fillna(0)
        .astype(int)
    )

    combo_gdf["past_year_unique_events_per_1000"] = np.where(
        combo_gdf["population"].notna()
        & (combo_gdf["population"] > 0),
        combo_gdf["past_year_unique_events"] / combo_gdf["population"] * 1000,
        np.nan,
    )

    fig_map.add_trace(
        go.Choroplethmapbox(
            geojson=base_geojson,
            locations=combo_gdf["plot_feature_id"],
            z=combo_gdf["past_year_unique_events_per_1000"],
            featureidkey="properties.plot_feature_id",
            colorscale="Viridis",
            marker={
                "opacity": 0.68,
                "line": {
                    "width": 0.4,
                    "color": "rgba(255,255,255,0.35)",
                },
            },
            colorbar={
                "title": "Selected-bin events<br>per 1,000 residents",
                "x": 0.98,
                "y": 0.50,
                "len": 0.62,
            },
            name="Past-year events per 1,000 residents",
            visible=combo_visible,
            customdata=combo_gdf[
                [
                    "mcpp_neighborhood_display",
                    "population",
                    "past_year_unique_events_per_1000",
                    "past_year_unique_events",
                    "selected_median_response_minutes",
                ]
            ].to_numpy(),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                f"Selected importance bin(s): {combo_label}<br>"
                "Population: %{customdata[1]:,.0f}<br>"
                "<br>"
                "Past-year unique CAD events in selected bin(s): %{customdata[3]:,}<br>"
                "Past-year selected-bin events per 1,000 residents: %{customdata[2]:.1f}<br>"
                "Median response time for selected bin(s): %{customdata[4]:.1f} min"
                "<extra></extra>"
            ),
        )
    )

    trace_metadata.append({
        "combo_label": combo_label,
        "trace_type": "choropleth",
    })

    # --------------------------------------------------------
    # Point layer:
    # Add neighborhood-level selected-bin metrics to point hover.
    # This is the key fix for the NaN hover issue.
    # --------------------------------------------------------

    point_metric_lookup = combo_gdf[
        [
            "mcpp_neighborhood",
            "population",
            "past_year_unique_events",
            "past_year_unique_events_per_1000",
            "selected_median_response_minutes",
        ]
    ].copy()

    point_metric_lookup["mcpp_neighborhood"] = (
        point_metric_lookup["mcpp_neighborhood"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    combo_points = point_events[
        point_events["event_importance_bin"].isin(bin_combo)
    ].copy()

    combo_points["mcpp_neighborhood"] = (
        combo_points["mcpp_neighborhood"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    combo_points = combo_points.merge(
        point_metric_lookup,
        on="mcpp_neighborhood",
        how="left",
    )

    combo_points["mcpp_neighborhood_display"] = (
        combo_points["mcpp_neighborhood"]
        .astype("string")
        .str.title()
    )

    combo_points["population_display"] = np.where(
        combo_points["population"].notna(),
        combo_points["population"].round(0).astype("Int64").astype(str),
        "Not available",
    )

    combo_points["population_display"] = (
        combo_points["population_display"]
        .replace("<NA>", "Not available")
    )

    combo_points["past_year_unique_events_display"] = np.where(
        combo_points["past_year_unique_events"].notna(),
        combo_points["past_year_unique_events"].round(0).astype("Int64").astype(str),
        "Not available",
    )

    combo_points["past_year_unique_events_display"] = (
        combo_points["past_year_unique_events_display"]
        .replace("<NA>", "Not available")
    )

    combo_points["past_year_events_per_1000_display"] = np.where(
        combo_points["past_year_unique_events_per_1000"].notna(),
        combo_points["past_year_unique_events_per_1000"].round(1).astype(str),
        "Not available",
    )

    combo_points["median_response_display"] = np.where(
        combo_points["selected_median_response_minutes"].notna(),
        combo_points["selected_median_response_minutes"].round(1).astype(str) + " min",
        "Not available",
    )

    for bin_name in TARGET_IMPORTANCE_BINS:
        bin_points = combo_points[
            combo_points["event_importance_bin"] == bin_name
        ].copy()

        if bin_points.empty:
            continue

        fig_map.add_trace(
            go.Scattermapbox(
                lat=bin_points[LAT_COL],
                lon=bin_points[LON_COL],
                mode="markers",
                name=bin_name,
                legendgroup=bin_name,
                showlegend=True,
                visible=combo_visible,
                marker=dict(
                    size=8,
                    opacity=0.85,
                ),
                customdata=bin_points[
                    [
                        "event_date_for_filter",
                        EVENT_ID_COLUMN,
                        "event_time_display",
                        "event_importance_bin",
                        "event_group",
                        "priority",
                        "initial_call_type",
                        "final_call_type",
                        "mcpp_neighborhood_display",
                        "population_display",
                        "past_year_unique_events_display",
                        "past_year_events_per_1000_display",
                        "median_response_display",
                    ]
                ].to_numpy(),
                hovertemplate=(
                    "<b>CAD Event:</b> %{customdata[1]}<br>"
                    "<b>Time:</b> %{customdata[2]}<br>"
                    f"<b>Selected importance bin(s):</b> {combo_label}<br>"
                    "<b>Point bin:</b> %{customdata[3]}<br>"
                    "<b>Event group:</b> %{customdata[4]}<br>"
                    "<b>Priority:</b> %{customdata[5]}<br>"
                    "<br>"
                    "<b>Initial call type:</b> %{customdata[6]}<br>"
                    "<b>Final call type:</b> %{customdata[7]}<br>"
                    "<b>Neighborhood:</b> %{customdata[8]}<br>"
                    "<br>"
                    "<b>Population:</b> %{customdata[9]}<br>"
                    "<b>Past-year unique CAD events in selected bin(s):</b> %{customdata[10]}<br>"
                    "<b>Past-year selected-bin events per 1,000 residents:</b> %{customdata[11]}<br>"
                    "<b>Median response time for selected bin(s):</b> %{customdata[12]}"
                    "<extra></extra>"
                ),
            )
        )

        trace_metadata.append({
            "combo_label": combo_label,
            "trace_type": "points",
        })

# ------------------------------------------------------------
# Dropdown buttons
# ------------------------------------------------------------

buttons = []

for bin_combo in bin_combinations:
    combo_label = make_bin_combo_label(bin_combo)

    visibility = [
        metadata["combo_label"] == combo_label
        for metadata in trace_metadata
    ]

    buttons.append(
        dict(
            label=combo_label,
            method="update",
            args=[
                {"visible": visibility},
                {
                    "title": (
                        "Past-Year Unique CAD Events per 1,000 Residents "
                        f"by MCPP Neighborhood ({combo_label})"
                    )
                },
            ],
        )
    )

# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

fig_map.update_layout(
    title=(
        "Past-Year Unique CAD Events per 1,000 Residents "
        "by MCPP Neighborhood (All selected bins)"
    ),
    mapbox=dict(
        style=PLOTLY_MAP_STYLE,
        center=PLOTLY_SEATTLE_CENTER,
        zoom=10,
    ),
    margin={
        "r": 15,
        "t": 72,
        "l": 0,
        "b": 0,
    },
    legend=dict(
        title_text="Latest-day points",
        x=0.07,
        y=0.58,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(17,17,17,0.72)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
    ),
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.01,
            y=0.98,
            xanchor="left",
            yanchor="top",
            buttons=buttons,
            showactive=True,
            bgcolor="rgba(17,17,17,0.85)",
            bordercolor="rgba(255,255,255,0.35)",
        )
    ],
)

fig_map.update_layout(
    title=(
        "Past-Year Unique CAD Events per 1,000 Residents "
        "by MCPP Neighborhood (All selected bins)"
    ),
    mapbox=dict(
        style=PLOTLY_MAP_STYLE,   # should be "carto-darkmatter"
        center=PLOTLY_SEATTLE_CENTER,
        zoom=10,
    ),
    template=PLOTLY_TEMPLATE,
    paper_bgcolor=PAPER_BG,
    plot_bgcolor=PAPER_BG,
    margin={
        "r": 15,
        "t": 72,
        "l": 0,
        "b": 0,
    },
    legend=dict(
        title_text="Latest-day points",
        x=0.07,
        y=0.58,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(17,17,17,0.72)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color="white"),
        title_font=dict(color="white"),
    ),
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.01,
            y=0.98,
            xanchor="left",
            yanchor="top",
            buttons=buttons,
            showactive=True,
            bgcolor="rgba(17,17,17,0.85)",
            bordercolor="rgba(255,255,255,0.35)",
            font=dict(color="white"),
        )
    ],
)

fig_map.show()

In [ ]:
# ------------------------------------------------------------
# Fig 2 - Daily Time Series
# Full past-year window available, initially zoomed to last 30 days
# All relative windows are anchored to latest available dataset date,
# not today's date.
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ------------------------------------------------------------
# Prep daily event data
# ------------------------------------------------------------

valid_time_bins = ensure_event_importance_bin(valid_time)

valid_time_bins[TIME_COLUMN] = pd.to_datetime(
    valid_time_bins[TIME_COLUMN],
    errors="coerce",
)

valid_time_bins = valid_time_bins[
    valid_time_bins[TIME_COLUMN].notna()
    & valid_time_bins[EVENT_ID_COLUMN].notna()
].copy()

valid_time_bins["date"] = (
    valid_time_bins[TIME_COLUMN]
    .dt.normalize()
)

# ------------------------------------------------------------
# Dataset-relative time windows
# ------------------------------------------------------------

latest_available_day = valid_time_bins["date"].max()

# Drop the earliest edge day because rolling snapshots often start mid-day,
# which can skew the first observed daily count and the early 7-day average.
earliest_available_day = valid_time_bins["date"].min()
earliest_analysis_day = earliest_available_day + pd.Timedelta(days=1)

past_year_start = latest_available_day - pd.Timedelta(days=364)

plot_start_day = max(
    earliest_analysis_day,
    past_year_start,
)

plot_end_day = latest_available_day

# Initial view = last 30 days ending at latest available dataset date
initial_view_start = latest_available_day - pd.Timedelta(days=29)

valid_time_bins = valid_time_bins[
    valid_time_bins["date"].between(plot_start_day, plot_end_day)
].copy()

date_index = pd.date_range(
    plot_start_day,
    plot_end_day,
    freq="D",
)

print(f"Full figure window: {plot_start_day.date()} to {plot_end_day.date()}")
print(f"Initial visible window: {initial_view_start.date()} to {plot_end_day.date()}")

# ------------------------------------------------------------
# Build dropdown figure
# ------------------------------------------------------------

fig_daily = go.Figure()

trace_metadata = []

for combo_i, bin_combo in enumerate(bin_combinations):
    combo_label = make_bin_combo_label(bin_combo)
    combo_visible = combo_i == 0

    combo_events = valid_time_bins[
        valid_time_bins["event_importance_bin"].isin(bin_combo)
    ].copy()

    daily_volume_combo = (
        combo_events
        .groupby("date")
        .agg(
            unique_call_events=(EVENT_ID_COLUMN, "nunique"),
            dispatch_records=(ROW_ID_COLUMN, "nunique"),
        )
        .reindex(date_index, fill_value=0)
        .rename_axis("date")
        .reset_index()
        .sort_values("date")
    )

    # min_periods=7 prevents the first few partial rolling averages
    # from visually distorting the scale.
    daily_volume_combo["unique_call_events_7d_avg"] = (
        daily_volume_combo["unique_call_events"]
        .rolling(window=7, min_periods=7)
        .mean()
    )

    fig_daily.add_trace(
        go.Scatter(
            x=daily_volume_combo["date"],
            y=daily_volume_combo["unique_call_events"],
            mode="lines",
            name="Daily events",
            visible=combo_visible,
            hovertemplate=(
                "<b>%{x|%Y-%m-%d}</b><br>"
                f"Selected importance bin(s): {combo_label}<br>"
                "Daily unique CAD events: %{y:,}"
                "<extra></extra>"
            ),
        )
    )

    trace_metadata.append({
        "combo_label": combo_label,
        "trace_type": "daily_events",
    })

    fig_daily.add_trace(
        go.Scatter(
            x=daily_volume_combo["date"],
            y=daily_volume_combo["unique_call_events_7d_avg"],
            mode="lines",
            name="7-day average",
            visible=combo_visible,
            hovertemplate=(
                "<b>%{x|%Y-%m-%d}</b><br>"
                f"Selected importance bin(s): {combo_label}<br>"
                "7-day average unique CAD events: %{y:,.1f}"
                "<extra></extra>"
            ),
        )
    )

    trace_metadata.append({
        "combo_label": combo_label,
        "trace_type": "seven_day_average",
    })

# ------------------------------------------------------------
# Dropdown buttons
# ------------------------------------------------------------

buttons = []

for bin_combo in bin_combinations:
    combo_label = make_bin_combo_label(bin_combo)

    visibility = [
        metadata["combo_label"] == combo_label
        for metadata in trace_metadata
    ]

    buttons.append(
        dict(
            label=combo_label,
            method="update",
            args=[
                {"visible": visibility},
                {
                    "title": (
                        "Daily SPD Unique CAD Events with 7-Day Average "
                        f"({combo_label})"
                    ),
                    "xaxis": {
                        "range": [
                            initial_view_start,
                            plot_end_day,
                        ],
                        "rangeslider": {
                            "visible": True,
                            "thickness": 0.08,
                        },
                    },
                },
            ],
        )
    )

# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

fig_daily.update_layout(
    title="Daily SPD Unique CAD Events with 7-Day Average (All selected bins)",
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    xaxis_title="Date",
    yaxis_title="Unique CAD events",
    legend_title_text="Metric",
    xaxis=dict(
        range=[
            initial_view_start,
            plot_end_day,
        ],
        rangeslider=dict(
            visible=True,
            thickness=0.08,
        ),
    ),
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=1.02,
            y=1.15,
            xanchor="left",
            yanchor="top",
            buttons=buttons,
            showactive=True,
            bgcolor="rgba(17,17,17,0.85)",
            bordercolor="rgba(255,255,255,0.35)",
        )
    ],
    margin={"l": 50, "r": 190, "t": 75, "b": 50},
)

fig_daily.add_annotation(
    text="Importance bins",
    x=1.02,
    y=1.28,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="left",
    font=dict(size=13),
)

fig_daily.add_annotation(
    text=(
        f"Initial view shows last 30 days available in data: "
        f"{initial_view_start.date()} to {plot_end_day.date()}"
    ),
    x=0,
    y=-0.22,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="left",
    font=dict(size=11),
)

fig_daily.show()

In [ ]:
# ------------------------------------------------------------
# Fig 3 - Scatter Plot
# Per-capita call volume vs. median response time
# Dropdown controls event importance bin(s)
# X-axis title kept centered and visible
# ------------------------------------------------------------

import itertools
import numpy as np
import pandas as pd
import plotly.graph_objects as go

TARGET_IMPORTANCE_BINS = [
    "property/nonviolent",
    "drug-related",
    "violent/person crime",
]

MIN_EVENTS_FOR_SCATTER = 100

# ------------------------------------------------------------
# Prepare response data
# ------------------------------------------------------------

scatter_response = response_analysis[
    response_analysis["event_importance_bin"].isin(TARGET_IMPORTANCE_BINS)
].copy()

scatter_response = scatter_response[
    ~scatter_response["dispatch_neighborhood"].isin(
        ["unknown", "-", "", "nan"]
    )
].copy()

volume_response_scatter = (
    scatter_response
    .groupby(["dispatch_neighborhood", "event_importance_bin"], as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        median_response_minutes=("response_time_minutes", "median"),
        mean_response_minutes=("response_time_minutes", "mean"),
        p75_response_minutes=("response_time_minutes", lambda x: x.quantile(0.75)),
        p90_response_minutes=("response_time_minutes", lambda x: x.quantile(0.90)),
    )
)

volume_response_scatter = volume_response_scatter[
    volume_response_scatter["unique_call_events"] >= MIN_EVENTS_FOR_SCATTER
].copy()

volume_response_scatter = volume_response_scatter.merge(
    neighborhood_population,
    on="dispatch_neighborhood",
    how="left",
)

volume_response_scatter = volume_response_scatter[
    volume_response_scatter["population"].notna()
    & (volume_response_scatter["population"] > 0)
].copy()

volume_response_scatter["annualized_events_per_1000"] = (
    volume_response_scatter["unique_call_events"]
    / years_observed
    / volume_response_scatter["population"]
    * 1000
)

volume_response_scatter = volume_response_scatter[
    volume_response_scatter["annualized_events_per_1000"] > 0
].copy()

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def make_bin_combo_label(bin_combo):
    if len(bin_combo) == len(TARGET_IMPORTANCE_BINS):
        return "All selected bins"
    return " + ".join(bin_combo)


def add_concern_score(data):
    out = data.copy()

    out["volume_rank"] = (
        out["annualized_events_per_1000"]
        .rank(pct=True)
    )

    out["response_rank"] = (
        out["median_response_minutes"]
        .rank(pct=True)
    )

    out["concern_score"] = (
        out["volume_rank"]
        * out["response_rank"]
    )

    out["marker_size"] = (
        10 + 40 * out["concern_score"]
    )

    return out


bin_combinations = []

for r in range(1, len(TARGET_IMPORTANCE_BINS) + 1):
    for combo in itertools.combinations(TARGET_IMPORTANCE_BINS, r):
        bin_combinations.append(list(combo))

bin_combinations = [
    TARGET_IMPORTANCE_BINS
] + [
    combo for combo in bin_combinations
    if combo != TARGET_IMPORTANCE_BINS
]

# ------------------------------------------------------------
# Build figure
# ------------------------------------------------------------

fig_scatter = go.Figure()

trace_metadata = []

for combo_i, bin_combo in enumerate(bin_combinations):
    combo_label = make_bin_combo_label(bin_combo)
    combo_visible = combo_i == 0

    combo_df = volume_response_scatter[
        volume_response_scatter["event_importance_bin"].isin(bin_combo)
    ].copy()

    combo_df = add_concern_score(combo_df)

    median_event_rate = combo_df["annualized_events_per_1000"].median()
    median_of_median_response = combo_df["median_response_minutes"].median()

    for bin_name in TARGET_IMPORTANCE_BINS:
        bin_df = combo_df[
            combo_df["event_importance_bin"] == bin_name
        ].copy()

        if bin_df.empty:
            continue

        customdata = np.stack(
            [
                bin_df["dispatch_neighborhood"].astype("string").str.title(),
                bin_df["event_importance_bin"],
                bin_df["population"],
                bin_df["unique_call_events"],
                bin_df["annualized_events_per_1000"],
                bin_df["median_response_minutes"],
                bin_df["mean_response_minutes"],
                bin_df["p75_response_minutes"],
                bin_df["p90_response_minutes"],
                bin_df["volume_rank"],
                bin_df["response_rank"],
                bin_df["concern_score"],
            ],
            axis=-1,
        )

        fig_scatter.add_trace(
            go.Scatter(
                x=bin_df["annualized_events_per_1000"],
                y=bin_df["median_response_minutes"],
                mode="markers",
                name=bin_name,
                legendgroup=bin_name,
                showlegend=combo_i == 0,
                visible=combo_visible,
                marker=dict(
                    size=bin_df["marker_size"],
                    sizemode="diameter",
                    opacity=0.75,
                    line=dict(width=0.8, color="white"),
                ),
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Selected importance bin: %{customdata[1]}<br>"
                    "Population: %{customdata[2]:,.0f}<br>"
                    "Raw unique CAD events: %{customdata[3]:,}<br>"
                    "Annualized events per 1,000 residents: %{customdata[4]:.1f}<br>"
                    "Median response: %{customdata[5]:.1f} min<br>"
                    "Mean response: %{customdata[6]:.1f} min<br>"
                    "P75 response: %{customdata[7]:.1f} min<br>"
                    "P90 response: %{customdata[8]:.1f} min<br>"
                    "<br>"
                    "Volume percentile rank: %{customdata[9]:.2f}<br>"
                    "Response percentile rank: %{customdata[10]:.2f}<br>"
                    "Concern score: %{customdata[11]:.2f}"
                    "<extra></extra>"
                ),
            )
        )

        trace_metadata.append({
            "combo_label": combo_label,
            "trace_type": "points",
        })

    fig_scatter.add_trace(
        go.Scatter(
            x=[median_event_rate, median_event_rate],
            y=[
                combo_df["median_response_minutes"].min(),
                combo_df["median_response_minutes"].max(),
            ],
            mode="lines",
            name=f"Median rate: {median_event_rate:.1f}",
            visible=combo_visible,
            showlegend=False,
            line=dict(dash="dash", width=2, color="white"),
            hovertemplate=(
                f"Median annualized event rate: {median_event_rate:.1f} per 1,000 residents"
                "<extra></extra>"
            ),
        )
    )

    trace_metadata.append({
        "combo_label": combo_label,
        "trace_type": "vertical_median",
    })

    fig_scatter.add_trace(
        go.Scatter(
            x=[
                combo_df["annualized_events_per_1000"].min(),
                combo_df["annualized_events_per_1000"].max(),
            ],
            y=[median_of_median_response, median_of_median_response],
            mode="lines",
            name=f"Median response: {median_of_median_response:.1f} min",
            visible=combo_visible,
            showlegend=False,
            line=dict(dash="dash", width=2, color="white"),
            hovertemplate=(
                f"Median response time: {median_of_median_response:.1f} min"
                "<extra></extra>"
            ),
        )
    )

    trace_metadata.append({
        "combo_label": combo_label,
        "trace_type": "horizontal_median",
    })

# ------------------------------------------------------------
# Dropdown buttons
# ------------------------------------------------------------

buttons = []

for bin_combo in bin_combinations:
    combo_label = make_bin_combo_label(bin_combo)

    visibility = [
        metadata["combo_label"] == combo_label
        for metadata in trace_metadata
    ]

    buttons.append(
        dict(
            label=combo_label,
            method="update",
            args=[
                {"visible": visibility},
                {
                    "title": (
                        "Per-Capita Call Volume vs. Median SPD Response Time "
                        f"({combo_label})"
                    )
                },
            ],
        )
    )

# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

fig_scatter.update_layout(
    title="Per-Capita Call Volume vs. Median SPD Response Time (All selected bins)",
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,

    xaxis=dict(
        title=dict(
            text="Annualized unique CAD events per 1,000 residents (log scale)",
            standoff=18,
        ),
        type="log",
        automargin=True,
    ),

    yaxis=dict(
        title=dict(
            text="Median response time minutes",
            standoff=12,
        ),
        automargin=True,
    ),

    legend_title_text="Event importance bin",

    height=700,

    margin={
        "l": 90,
        "r": 300,
        "t": 120,
        "b": 105,
    },

    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=1.02,
            y=1.04,
            xanchor="left",
            yanchor="top",
            buttons=buttons,
            showactive=True,
            bgcolor="rgba(17,17,17,0.85)",
            bordercolor="rgba(255,255,255,0.35)",
        )
    ],

    legend=dict(
        x=1.02,
        y=0.86,
        xanchor="left",
        yanchor="top",
    ),
)

fig_scatter.add_annotation(
    text="Bin selection",
    x=1.02,
    y=1.10,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="left",
    font=dict(size=13),
)

fig_scatter.show()

PICKUP POINT - Make notes and fix Nans in map

In [ ]:
from pathlib import Path
import webbrowser
import json
import numpy as np
import pandas as pd
import plotly.io as pio
from plotly.utils import PlotlyJSONEncoder

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

dashboard_path = REPORTS_DIR / "dashboard_mockup.html"

# ------------------------------------------------------------
# Extract existing per-figure dropdown visibility settings.
# These dropdowns stay in the figures, but will be hidden/off-screen.
# ------------------------------------------------------------

def extract_dropdown_visibility(fig, figure_name):
    if not fig.layout.updatemenus:
        raise ValueError(
            f"{figure_name} does not have an updatemenu/dropdown. "
            "Rerun that figure cell before running this dashboard export cell."
        )

    menu = fig.layout.updatemenus[0]
    visibility_lookup = {}

    for button in menu.buttons:
        label = str(button.label)
        visible_array = list(button.args[0]["visible"])
        visibility_lookup[label] = visible_array

    return visibility_lookup


map_visibility = extract_dropdown_visibility(fig_map, "fig_map")
daily_visibility = extract_dropdown_visibility(fig_daily, "fig_daily")
scatter_visibility = extract_dropdown_visibility(fig_scatter, "fig_scatter")

combo_labels = list(scatter_visibility.keys())
default_combo_label = combo_labels[0]

missing_from_map = set(combo_labels) - set(map_visibility.keys())
missing_from_daily = set(combo_labels) - set(daily_visibility.keys())

if missing_from_map:
    raise ValueError(f"fig_map is missing dropdown labels: {missing_from_map}")

if missing_from_daily:
    raise ValueError(f"fig_daily is missing dropdown labels: {missing_from_daily}")


# ------------------------------------------------------------
# Set initial selected traces
# ------------------------------------------------------------

def set_initial_visibility(fig, visibility_lookup, label):
    visible_array = visibility_lookup[label]

    for trace, visible in zip(fig.data, visible_array):
        trace.visible = visible


set_initial_visibility(fig_map, map_visibility, default_combo_label)
set_initial_visibility(fig_daily, daily_visibility, default_combo_label)
set_initial_visibility(fig_scatter, scatter_visibility, default_combo_label)


# ------------------------------------------------------------
# Build a Python-side payload of all map point trace data.
# This avoids relying on browser-side trace caching.
# ------------------------------------------------------------

def to_plain_list(values):
    if values is None:
        return []

    if isinstance(values, np.ndarray):
        return values.tolist()

    return list(values)


def customdata_to_plain_rows(customdata):
    rows = to_plain_list(customdata)
    plain_rows = []

    for row in rows:
        if isinstance(row, np.ndarray):
            plain_rows.append(row.tolist())
        elif isinstance(row, (list, tuple)):
            plain_rows.append(list(row))
        else:
            plain_rows.append([row])

    return plain_rows


def date_col_is_valid(customdata_rows, col_idx):
    if not customdata_rows:
        return False

    values = [
        row[col_idx]
        for row in customdata_rows[:50]
        if len(row) > col_idx
    ]

    if not values:
        return False

    parsed = pd.to_datetime(
        pd.Series(values),
        errors="coerce",
    )

    plausible_dates = (
        parsed.notna()
        & (parsed >= pd.Timestamp("2008-01-01"))
        & (parsed <= pd.Timestamp("2035-01-01"))
    )

    return plausible_dates.sum() >= max(1, int(len(values) * 0.80))


def guess_customdata_date_col(customdata_rows):
    # Updated Fig 1 should use customdata[0] = event_date_for_filter.
    if date_col_is_valid(customdata_rows, 0):
        return 0

    # Fallback for older point hover formats.
    max_cols = min(
        max(len(row) for row in customdata_rows[:50]),
        6,
    )

    for col_idx in range(max_cols):
        if date_col_is_valid(customdata_rows, col_idx):
            return col_idx

    return None


def is_map_point_trace(trace):
    trace_type = str(getattr(trace, "type", "")).lower()
    trace_name = str(getattr(trace, "name", ""))

    return (
        trace_type in ["scattermapbox", "scattermap"]
        and trace_name in TARGET_IMPORTANCE_BINS
        and hasattr(trace, "lat")
        and hasattr(trace, "lon")
    )


map_point_payload = []

for trace_index, trace in enumerate(fig_map.data):
    if not is_map_point_trace(trace):
        continue

    lat_values = to_plain_list(trace.lat)
    lon_values = to_plain_list(trace.lon)
    customdata_rows = customdata_to_plain_rows(trace.customdata)

    date_col_idx = guess_customdata_date_col(customdata_rows)

    if date_col_idx is None:
        raise ValueError(
            f"Could not detect a date column in map point trace {trace_index} "
            f"({trace.name}). Rerun the updated Fig 1 cell where "
            "customdata[0] is event_date_for_filter."
        )

    map_point_payload.append(
        {
            "traceIndex": trace_index,
            "name": str(trace.name),
            "lat": lat_values,
            "lon": lon_values,
            "customdata": customdata_rows,
            "dateColumnIndex": date_col_idx,
        }
    )

total_cached_points = sum(
    len(trace_payload["lat"])
    for trace_payload in map_point_payload
)

print(f"Map point traces cached: {len(map_point_payload):,}")
print(f"Map points cached: {total_cached_points:,}")
print(
    "Detected map point date columns:",
    [
        {
            "trace": payload["name"],
            "traceIndex": payload["traceIndex"],
            "dateColumnIndex": payload["dateColumnIndex"],
            "points": len(payload["lat"]),
        }
        for payload in map_point_payload
    ],
)

if len(map_point_payload) == 0:
    raise ValueError(
        "No map point traces were found in fig_map. "
        "Rerun the Fig 1 map cell and make sure it creates Scattermapbox point traces."
    )

if total_cached_points == 0:
    raise ValueError(
        "Map point traces were found, but they contain 0 points. "
        "Rerun the Fig 1 cell after updating it so point_events uses the full past-year point universe."
    )


# ------------------------------------------------------------
# Get default Fig 2 visible date range for initial map point filter.
# Prefer explicit xaxis range; otherwise fall back to latest 30 days in Fig 2 data.
# ------------------------------------------------------------

def date_string(value):
    if value is None:
        return None

    value = pd.to_datetime(value, errors="coerce")

    if pd.isna(value):
        return None

    return value.strftime("%Y-%m-%d")


fig_daily_xaxis_range = fig_daily.layout.xaxis.range

if fig_daily_xaxis_range and len(fig_daily_xaxis_range) >= 2:
    default_daily_start = date_string(fig_daily_xaxis_range[0])
    default_daily_end = date_string(fig_daily_xaxis_range[1])
else:
    daily_dates = []

    for trace in fig_daily.data:
        if hasattr(trace, "x") and trace.x is not None:
            parsed_dates = pd.to_datetime(
                pd.Series(to_plain_list(trace.x)),
                errors="coerce",
            ).dropna()

            daily_dates.extend(parsed_dates.tolist())

    if len(daily_dates) == 0:
        raise ValueError(
            "Could not infer a default Fig 2 date range. "
            "Rerun the Fig 2 cell before exporting."
        )

    latest_daily_date = max(daily_dates).normalize()
    default_daily_start = (latest_daily_date - pd.Timedelta(days=29)).strftime("%Y-%m-%d")
    default_daily_end = latest_daily_date.strftime("%Y-%m-%d")

print(f"Default map point filter from Fig 2: {default_daily_start} to {default_daily_end}")


# ------------------------------------------------------------
# Hide plot-specific dropdowns without removing them.
# Keep x/y inside Plotly's accepted range.
# ------------------------------------------------------------

def hide_plot_specific_dropdowns(fig):
    hidden_menus = []

    for menu in fig.layout.updatemenus:
        menu_dict = menu.to_plotly_json()

        menu_dict.update(
            {
                "visible": False,
                "x": -2.0,
                "y": 3.0,
                "xanchor": "left",
                "yanchor": "top",
            }
        )

        hidden_menus.append(menu_dict)

    fig.update_layout(updatemenus=hidden_menus)


hide_plot_specific_dropdowns(fig_map)
hide_plot_specific_dropdowns(fig_daily)
hide_plot_specific_dropdowns(fig_scatter)


# ------------------------------------------------------------
# Trace-level legend eligibility
# These decide which traces are allowed to appear if a legend is shown.
#
# For fig_map:
# - The point legend is intentionally hidden at the layout level.
# - The sidebar map control targets the choropleth colorbar instead.
# ------------------------------------------------------------

for trace in fig_map.data:
    trace.showlegend = trace.name in TARGET_IMPORTANCE_BINS

for trace in fig_daily.data:
    trace.showlegend = trace.name in ["Daily events", "7-day average"]

for trace in fig_scatter.data:
    trace.showlegend = trace.name in TARGET_IMPORTANCE_BINS


# ------------------------------------------------------------
# Map choropleth colorbar control
# The map "legend" desired here is the choropleth color scale/colorbar,
# controlled by trace.showscale, not layout.showlegend.
# ------------------------------------------------------------

def is_map_choropleth_trace(trace):
    trace_type = str(getattr(trace, "type", "")).lower()

    return trace_type in [
        "choroplethmapbox",
        "choroplethmap",
    ]


map_choropleth_trace_indices = []

for trace_index, trace in enumerate(fig_map.data):
    if is_map_choropleth_trace(trace):
        map_choropleth_trace_indices.append(trace_index)

        # Hide choropleth colorbar by default.
        trace.showscale = False

print(f"Map choropleth colorbar traces: {map_choropleth_trace_indices}")


# ------------------------------------------------------------
# Browser dashboard sizing
# Legends/colorbar hidden by default.
# ------------------------------------------------------------

fig_map.update_layout(
    autosize=True,
    height=None,
    width=None,
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
    showlegend=False,  # keeps point legend hidden
    legend=dict(
        title_text="Map points",
        x=0.07,
        y=0.58,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(17,17,17,0.72)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
    ),
)

fig_daily.update_layout(
    autosize=True,
    height=None,
    width=None,
    margin={"l": 50, "r": 35, "t": 55, "b": 45},
    showlegend=False,
)

fig_scatter.update_xaxes(
    title_text="",
    automargin=True,
)

fig_scatter.update_layout(
    autosize=True,
    height=None,
    width=None,
    margin={"l": 90, "r": 35, "t": 70, "b": 45},
    showlegend=False,
)


# ------------------------------------------------------------
# Convert figures to HTML
# ------------------------------------------------------------

map_html = pio.to_html(
    fig_map,
    full_html=False,
    include_plotlyjs=True,
    default_width="100%",
    default_height="100%",
    config={"responsive": True},
)

daily_html = pio.to_html(
    fig_daily,
    full_html=False,
    include_plotlyjs=False,
    default_width="100%",
    default_height="100%",
    config={"responsive": True},
)

scatter_html = pio.to_html(
    fig_scatter,
    full_html=False,
    include_plotlyjs=False,
    default_width="100%",
    default_height="100%",
    config={"responsive": True},
)


# ------------------------------------------------------------
# Global dropdown data for JavaScript
# ------------------------------------------------------------

global_visibility_payload = {
    "map": map_visibility,
    "daily": daily_visibility,
    "scatter": scatter_visibility,
}

global_title_payload = {
    "map": {
        label: (
            "Past-Year Unique CAD Events per 1,000 Residents "
            f"by MCPP Neighborhood ({label})"
        )
        for label in combo_labels
    },
    "daily": {
        label: (
            "Daily SPD Unique CAD Events with 7-Day Average "
            f"({label})"
        )
        for label in combo_labels
    },
    "scatter": {
        label: (
            "Per-Capita Call Volume vs. Median SPD Response Time "
            f"({label})"
        )
        for label in combo_labels
    },
}

global_visibility_json = json.dumps(
    global_visibility_payload,
    cls=PlotlyJSONEncoder,
)

global_title_json = json.dumps(
    global_title_payload,
    cls=PlotlyJSONEncoder,
)

combo_labels_json = json.dumps(
    combo_labels,
    cls=PlotlyJSONEncoder,
)

default_combo_label_json = json.dumps(
    default_combo_label,
    cls=PlotlyJSONEncoder,
)

map_point_payload_json = json.dumps(
    map_point_payload,
    cls=PlotlyJSONEncoder,
)

default_daily_range_json = json.dumps(
    [default_daily_start, default_daily_end],
    cls=PlotlyJSONEncoder,
)

map_choropleth_trace_indices_json = json.dumps(
    map_choropleth_trace_indices,
    cls=PlotlyJSONEncoder,
)


# ------------------------------------------------------------
# Full browser dashboard
# ------------------------------------------------------------

full_dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>SPD Dashboard Mockup</title>

    <style>
        html, body {{
            margin: 0;
            padding: 0;
            width: 100%;
            height: 100%;
            background-color: #111111;
            overflow: hidden;
            font-family: Arial, sans-serif;
        }}

        .dashboard-shell {{
            width: 100vw;
            height: 100vh;
            display: grid;
            grid-template-rows: 42px 1fr;
            background-color: #111111;
        }}

        .dashboard-controls {{
            display: flex;
            align-items: center;
            gap: 12px;
            padding: 6px 12px;
            box-sizing: border-box;
            border-bottom: 1px solid #333;
            background-color: #151515;
            color: white;
        }}

        .dashboard-controls label {{
            font-size: 13px;
            color: #d6d6d6;
        }}

        .dashboard-controls select {{
            background-color: #222;
            color: white;
            border: 1px solid #555;
            border-radius: 6px;
            padding: 4px 8px;
            font-size: 13px;
        }}

        .dashboard-controls .hint {{
            color: #aaa;
            font-size: 12px;
        }}

        .dashboard-controls .map-window {{
            color: #bbb;
            font-size: 12px;
            margin-left: auto;
        }}

        .dashboard-grid {{
            display: grid;
            grid-template-columns: 1.15fr 1fr;
            grid-template-rows: 1fr 1fr;
            gap: 10px;
            width: 100%;
            height: 100%;
            background-color: #111111;
            padding: 10px;
            box-sizing: border-box;
            min-height: 0;
        }}

        .panel {{
            min-width: 0;
            min-height: 0;
            overflow: hidden;
            border: 1px solid #333;
            border-radius: 8px;
            background-color: #111111;
        }}

        .map-panel {{
            grid-column: 1;
            grid-row: 1 / span 2;
        }}

        .daily-panel {{
            grid-column: 2;
            grid-row: 1;
        }}

        .scatter-panel {{
            grid-column: 2;
            grid-row: 2;
        }}

        .map-panel .plotly-graph-div,
        .daily-panel .plotly-graph-div {{
            width: 100% !important;
            height: 100% !important;
        }}

        .scatter-panel-with-label {{
            position: relative;
        }}

        .scatter-plot-wrap {{
            width: 100%;
            height: calc(100% - 28px);
        }}

        .scatter-plot-wrap .plotly-graph-div {{
            width: 100% !important;
            height: 100% !important;
        }}

        .scatter-x-axis-title {{
            position: absolute;
            left: 0;
            right: 0;
            bottom: 5px;
            height: 20px;
            text-align: center;
            color: white;
            font-size: 13px;
            line-height: 20px;
            pointer-events: none;
            z-index: 10;
        }}

        .legend-sidebar-toggle {{
            position: fixed;
            right: 0;
            top: 50%;
            transform: translateY(-50%);
            z-index: 9999;
            background-color: #222;
            color: white;
            border: 1px solid #555;
            border-right: none;
            border-radius: 8px 0 0 8px;
            padding: 10px 8px;
            cursor: pointer;
            font-size: 13px;
        }}

        .legend-sidebar {{
            position: fixed;
            right: -300px;
            top: 42px;
            width: 300px;
            height: calc(100vh - 42px);
            z-index: 9998;
            background-color: rgba(17, 17, 17, 0.96);
            border-left: 1px solid #444;
            color: white;
            padding: 16px;
            box-sizing: border-box;
            transition: right 0.25s ease;
        }}

        .legend-sidebar.open {{
            right: 0;
        }}

        .legend-sidebar h3 {{
            margin-top: 0;
            font-size: 16px;
        }}

        .legend-sidebar p {{
            color: #aaa;
            font-size: 12px;
            line-height: 1.4;
            margin-bottom: 16px;
        }}

        .legend-sidebar label {{
            display: block;
            margin: 12px 0;
            font-size: 13px;
            color: #ddd;
            cursor: pointer;
        }}

        .legend-sidebar input {{
            margin-right: 8px;
        }}
    </style>
</head>

<body>
    <div class="dashboard-shell">

        <div class="dashboard-controls">
            <label for="importance-bin-select">
                Event importance bins:
            </label>

            <select id="importance-bin-select"></select>

            <span class="hint">
                This one control updates all figures. Fig 2's visible date range filters Fig 1 map points.
            </span>

            <span class="map-window" id="map-point-window-label">
                Map points: syncing...
            </span>
        </div>

        <div class="dashboard-grid">
            <div class="panel map-panel">
                {map_html}
            </div>

            <div class="panel daily-panel">
                {daily_html}
            </div>

            <div class="panel scatter-panel scatter-panel-with-label">
                <div class="scatter-plot-wrap">
                    {scatter_html}
                </div>

                <div class="scatter-x-axis-title">
                    Annualized unique CAD events per 1,000 residents, log scale
                </div>
            </div>
        </div>
    </div>

    <button class="legend-sidebar-toggle" id="legend-sidebar-toggle">
        Legends
    </button>

    <div class="legend-sidebar" id="legend-sidebar">
        <h3>Legend controls</h3>

        <p>
            Legends are hidden by default to keep the dashboard compact.
            The map control shows the choropleth color scale, not the point legend.
        </p>

        <label>
            <input type="checkbox" id="legend-all-toggle">
            Show all legends / color scale
        </label>

        <label>
            <input type="checkbox" id="legend-map-toggle">
            Show map color scale
        </label>

        <label>
            <input type="checkbox" id="legend-daily-toggle">
            Show daily chart legend
        </label>

        <label>
            <input type="checkbox" id="legend-scatter-toggle">
            Show scatterplot legend
        </label>
    </div>

    <script>
        const comboLabels = {combo_labels_json};
        const defaultComboLabel = {default_combo_label_json};

        const visibilityLookup = {global_visibility_json};
        const titleLookup = {global_title_json};

        const mapPointPayload = {map_point_payload_json};
        const defaultDailyRange = {default_daily_range_json};
        const mapChoroplethTraceIndices = {map_choropleth_trace_indices_json};

        function getPlots() {{
            return {{
                map: document.querySelector(".map-panel .js-plotly-plot"),
                daily: document.querySelector(".daily-panel .js-plotly-plot"),
                scatter: document.querySelector(".scatter-panel .js-plotly-plot")
            }};
        }}

        function resizePlots() {{
            if (!window.Plotly) {{
                return;
            }}

            document.querySelectorAll(".js-plotly-plot").forEach(function(plot) {{
                window.Plotly.Plots.resize(plot);
            }});
        }}

        function setPlotLegend(plot, showLegend) {{
            if (!plot || !window.Plotly) {{
                return Promise.resolve();
            }}

            return window.Plotly.relayout(
                plot,
                {{"showlegend": showLegend}}
            );
        }}

        function setMapColorScale(showScale) {{
            const plots = getPlots();

            if (
                !plots.map
                || !window.Plotly
                || !mapChoroplethTraceIndices
                || mapChoroplethTraceIndices.length === 0
            ) {{
                return Promise.resolve();
            }}

            return window.Plotly.restyle(
                plots.map,
                {{
                    showscale: Array(mapChoroplethTraceIndices.length).fill(showScale)
                }},
                mapChoroplethTraceIndices
            );
        }}

        function updateAllLegendCheckboxState() {{
            const allToggle = document.getElementById("legend-all-toggle");
            const mapToggle = document.getElementById("legend-map-toggle");
            const dailyToggle = document.getElementById("legend-daily-toggle");
            const scatterToggle = document.getElementById("legend-scatter-toggle");

            if (!allToggle || !mapToggle || !dailyToggle || !scatterToggle) {{
                return;
            }}

            allToggle.checked = (
                mapToggle.checked
                && dailyToggle.checked
                && scatterToggle.checked
            );
        }}

        function setAllLegends(showLegend) {{
            const plots = getPlots();

            const mapToggle = document.getElementById("legend-map-toggle");
            const dailyToggle = document.getElementById("legend-daily-toggle");
            const scatterToggle = document.getElementById("legend-scatter-toggle");
            const allToggle = document.getElementById("legend-all-toggle");

            if (mapToggle) {{
                mapToggle.checked = showLegend;
            }}

            if (dailyToggle) {{
                dailyToggle.checked = showLegend;
            }}

            if (scatterToggle) {{
                scatterToggle.checked = showLegend;
            }}

            if (allToggle) {{
                allToggle.checked = showLegend;
            }}

            return Promise.all([
                setMapColorScale(showLegend),
                setPlotLegend(plots.daily, showLegend),
                setPlotLegend(plots.scatter, showLegend),
            ]).then(function() {{
                resizePlots();
            }});
        }}

        function initializeLegendSidebar() {{
            const sidebar = document.getElementById("legend-sidebar");
            const sidebarToggle = document.getElementById("legend-sidebar-toggle");

            const allToggle = document.getElementById("legend-all-toggle");
            const mapToggle = document.getElementById("legend-map-toggle");
            const dailyToggle = document.getElementById("legend-daily-toggle");
            const scatterToggle = document.getElementById("legend-scatter-toggle");

            sidebarToggle.addEventListener("click", function() {{
                sidebar.classList.toggle("open");
            }});

            allToggle.addEventListener("change", function(event) {{
                setAllLegends(event.target.checked);
            }});

            mapToggle.addEventListener("change", function(event) {{
                setMapColorScale(event.target.checked).then(function() {{
                    updateAllLegendCheckboxState();
                    resizePlots();
                }});
            }});

            dailyToggle.addEventListener("change", function(event) {{
                const plots = getPlots();

                setPlotLegend(plots.daily, event.target.checked).then(function() {{
                    updateAllLegendCheckboxState();
                    resizePlots();
                }});
            }});

            scatterToggle.addEventListener("change", function(event) {{
                const plots = getPlots();

                setPlotLegend(plots.scatter, event.target.checked).then(function() {{
                    updateAllLegendCheckboxState();
                    resizePlots();
                }});
            }});

            setAllLegends(false);
        }}

        function dateOnlyString(value) {{
            if (value === null || value === undefined) {{
                return null;
            }}

            const raw = String(value);

            if (raw.length >= 10 && raw[4] === "-" && raw[7] === "-") {{
                return raw.slice(0, 10);
            }}

            const parsed = new Date(value);

            if (Number.isNaN(parsed.getTime())) {{
                return null;
            }}

            const year = parsed.getFullYear();
            const month = String(parsed.getMonth() + 1).padStart(2, "0");
            const day = String(parsed.getDate()).padStart(2, "0");

            return `${{year}}-${{month}}-${{day}}`;
        }}

        function dateOnlyToLocalMs(value, endOfDay=false) {{
            const cleanDate = dateOnlyString(value);

            if (!cleanDate) {{
                return null;
            }}

            const parts = cleanDate.split("-").map(Number);
            const year = parts[0];
            const month = parts[1] - 1;
            const day = parts[2];

            if (endOfDay) {{
                return new Date(year, month, day, 23, 59, 59, 999).getTime();
            }}

            return new Date(year, month, day, 0, 0, 0, 0).getTime();
        }}

        function getDailyVisibleRange() {{
            const plots = getPlots();

            if (
                plots.daily
                && plots.daily.layout
                && plots.daily.layout.xaxis
                && plots.daily.layout.xaxis.range
                && plots.daily.layout.xaxis.range.length >= 2
            ) {{
                return {{
                    start: plots.daily.layout.xaxis.range[0],
                    end: plots.daily.layout.xaxis.range[1]
                }};
            }}

            if (
                plots.daily
                && plots.daily._fullLayout
                && plots.daily._fullLayout.xaxis
                && plots.daily._fullLayout.xaxis.range
                && plots.daily._fullLayout.xaxis.range.length >= 2
            ) {{
                return {{
                    start: plots.daily._fullLayout.xaxis.range[0],
                    end: plots.daily._fullLayout.xaxis.range[1]
                }};
            }}

            return {{
                start: defaultDailyRange[0],
                end: defaultDailyRange[1]
            }};
        }}

        function setMapPointWindowLabel(startDate, endDate, visiblePointCount) {{
            const label = document.getElementById("map-point-window-label");

            if (!label) {{
                return;
            }}

            const startClean = dateOnlyString(startDate);
            const endClean = dateOnlyString(endDate);

            if (!startClean || !endClean) {{
                label.textContent = "Map points: date range unavailable";
                return;
            }}

            label.textContent = (
                "Map points: "
                + startClean
                + " to "
                + endClean
                + " | visible points: "
                + visiblePointCount.toLocaleString()
            );
        }}

        function filterMapPointsToDateRange(startDate, endDate) {{
            const plots = getPlots();

            if (!plots.map) {{
                return;
            }}

            const startMs = dateOnlyToLocalMs(startDate, false);
            const endMs = dateOnlyToLocalMs(endDate, true);

            if (startMs === null || endMs === null) {{
                return;
            }}

            let visiblePointCount = 0;
            const restyleUpdates = {{}};
            const traceIndices = [];

            mapPointPayload.forEach(function(payload) {{
                const traceIndex = payload.traceIndex;
                const dateColumnIndex = payload.dateColumnIndex;

                const filteredLat = [];
                const filteredLon = [];
                const filteredCustomdata = [];

                payload.customdata.forEach(function(row, i) {{
                    const eventDate = row[dateColumnIndex];
                    const eventMs = dateOnlyToLocalMs(eventDate, false);

                    if (
                        eventMs !== null
                        && eventMs >= startMs
                        && eventMs <= endMs
                    ) {{
                        filteredLat.push(payload.lat[i]);
                        filteredLon.push(payload.lon[i]);
                        filteredCustomdata.push(row);
                    }}
                }});

                visiblePointCount += filteredLat.length;

                traceIndices.push(traceIndex);

                if (!restyleUpdates.lat) {{
                    restyleUpdates.lat = [];
                    restyleUpdates.lon = [];
                    restyleUpdates.customdata = [];
                }}

                restyleUpdates.lat.push(filteredLat);
                restyleUpdates.lon.push(filteredLon);
                restyleUpdates.customdata.push(filteredCustomdata);
            }});

            if (traceIndices.length > 0) {{
                window.Plotly.restyle(
                    plots.map,
                    restyleUpdates,
                    traceIndices
                );
            }}

            setMapPointWindowLabel(startDate, endDate, visiblePointCount);
        }}

        function filterMapPointsToCurrentDailyRange() {{
            const range = getDailyVisibleRange();

            filterMapPointsToDateRange(range.start, range.end);
        }}

        function connectDailySliderToMapPoints() {{
            const plots = getPlots();

            if (!plots.daily) {{
                return;
            }}

            plots.daily.on("plotly_relayout", function(eventData) {{
                let startDate = null;
                let endDate = null;

                if (
                    eventData["xaxis.range[0]"] !== undefined
                    && eventData["xaxis.range[1]"] !== undefined
                ) {{
                    startDate = eventData["xaxis.range[0]"];
                    endDate = eventData["xaxis.range[1]"];
                }}

                else if (
                    eventData["xaxis.range"] !== undefined
                    && eventData["xaxis.range"].length >= 2
                ) {{
                    startDate = eventData["xaxis.range"][0];
                    endDate = eventData["xaxis.range"][1];
                }}

                else {{
                    const range = getDailyVisibleRange();

                    startDate = range.start;
                    endDate = range.end;
                }}

                if (startDate && endDate) {{
                    filterMapPointsToDateRange(startDate, endDate);
                }}
            }});
        }}

        function forceDailyDefaultRange() {{
            const plots = getPlots();

            if (!plots.daily || !window.Plotly) {{
                return Promise.resolve();
            }}

            return window.Plotly.relayout(
                plots.daily,
                {{
                    "xaxis.range": [
                        defaultDailyRange[0],
                        defaultDailyRange[1]
                    ]
                }}
            );
        }}

        function applyImportanceBinSelection(label) {{
            if (!window.Plotly) {{
                return Promise.resolve();
            }}

            const plots = getPlots();
            const updates = [];

            if (plots.map && visibilityLookup.map[label]) {{
                updates.push(
                    window.Plotly.update(
                        plots.map,
                        {{visible: visibilityLookup.map[label]}},
                        {{"title.text": titleLookup.map[label]}}
                    )
                );
            }}

            if (plots.daily && visibilityLookup.daily[label]) {{
                updates.push(
                    window.Plotly.update(
                        plots.daily,
                        {{visible: visibilityLookup.daily[label]}},
                        {{"title.text": titleLookup.daily[label]}}
                    )
                );
            }}

            if (plots.scatter && visibilityLookup.scatter[label]) {{
                updates.push(
                    window.Plotly.update(
                        plots.scatter,
                        {{visible: visibilityLookup.scatter[label]}},
                        {{"title.text": titleLookup.scatter[label]}}
                    )
                );
            }}

            return Promise.all(updates).then(function() {{
                const mapToggle = document.getElementById("legend-map-toggle");
                const mapColorScaleVisible = mapToggle ? mapToggle.checked : false;

                return setMapColorScale(mapColorScaleVisible);
            }}).then(function() {{
                filterMapPointsToCurrentDailyRange();
                resizePlots();
            }});
        }}

        function buildGlobalDropdown() {{
            const select = document.getElementById("importance-bin-select");

            comboLabels.forEach(function(label) {{
                const option = document.createElement("option");
                option.value = label;
                option.textContent = label;
                select.appendChild(option);
            }});

            select.value = defaultComboLabel;

            select.addEventListener("change", function(event) {{
                applyImportanceBinSelection(event.target.value);
            }});
        }}

        window.addEventListener("load", function() {{
            buildGlobalDropdown();
            initializeLegendSidebar();
            connectDailySliderToMapPoints();

            forceDailyDefaultRange()
                .then(function() {{
                    return applyImportanceBinSelection(defaultComboLabel);
                }})
                .then(function() {{
                    return setAllLegends(false);
                }})
                .then(function() {{
                    filterMapPointsToDateRange(
                        defaultDailyRange[0],
                        defaultDailyRange[1]
                    );
                    resizePlots();
                }});

            setTimeout(function() {{
                filterMapPointsToCurrentDailyRange();
                resizePlots();
            }}, 750);

            setTimeout(function() {{
                filterMapPointsToCurrentDailyRange();
                resizePlots();
            }}, 1500);

            setTimeout(function() {{
                filterMapPointsToCurrentDailyRange();
                resizePlots();
            }}, 2500);
        }});

        window.addEventListener("resize", resizePlots);
    </script>
</body>
</html>
"""

dashboard_path.write_text(full_dashboard_html, encoding="utf-8")

print(f"Saved dashboard to: {dashboard_path}")

webbrowser.open(dashboard_path.resolve().as_uri())

Old browser export cell

In [ ]:
from pathlib import Path
import webbrowser
import plotly.io as pio

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

dashboard_path = REPORTS_DIR / "dashboard_mockup.html"

# ------------------------------------------------------------
# Figure sizing for browser dashboard
# ------------------------------------------------------------

fig_map.update_layout(
    autosize=True,
    height=None,
    width=None,
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
)

fig_daily.update_layout(
    autosize=True,
    height=None,
    width=None,
    margin={"l": 50, "r": 170, "t": 55, "b": 45},
)

# Important:
# Remove Plotly's built-in x-axis title because it centers on the plot area,
# not the full dashboard panel.
fig_scatter.update_xaxes(
    title_text="",
    automargin=True,
)

fig_scatter.update_layout(
    autosize=True,
    height=None,
    width=None,
    margin={"l": 90, "r": 300, "t": 70, "b": 45},
)

# ------------------------------------------------------------
# Convert figures to HTML
# ------------------------------------------------------------

map_html = pio.to_html(
    fig_map,
    full_html=False,
    include_plotlyjs=True,
    default_width="100%",
    default_height="100%",
    config={"responsive": True},
)

daily_html = pio.to_html(
    fig_daily,
    full_html=False,
    include_plotlyjs=False,
    default_width="100%",
    default_height="100%",
    config={"responsive": True},
)

scatter_html = pio.to_html(
    fig_scatter,
    full_html=False,
    include_plotlyjs=False,
    default_width="100%",
    default_height="100%",
    config={"responsive": True},
)

# ------------------------------------------------------------
# Full browser dashboard
# ------------------------------------------------------------

full_dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>SPD Dashboard Mockup</title>

    <style>
        html, body {{
            margin: 0;
            padding: 0;
            width: 100%;
            height: 100%;
            background-color: #111111;
            overflow: hidden;
        }}

        .dashboard-grid {{
            display: grid;
            grid-template-columns: 1.15fr 1fr;
            grid-template-rows: 1fr 1fr;
            gap: 10px;
            width: 100vw;
            height: 100vh;
            background-color: #111111;
            padding: 10px;
            box-sizing: border-box;
        }}

        .panel {{
            min-width: 0;
            min-height: 0;
            overflow: hidden;
            border: 1px solid #333;
            border-radius: 8px;
            background-color: #111111;
        }}

        .map-panel {{
            grid-column: 1;
            grid-row: 1 / span 2;
        }}

        .daily-panel {{
            grid-column: 2;
            grid-row: 1;
        }}

        .scatter-panel {{
            grid-column: 2;
            grid-row: 2;
        }}

        .map-panel .plotly-graph-div,
        .daily-panel .plotly-graph-div {{
            width: 100% !important;
            height: 100% !important;
        }}

        .scatter-panel-with-label {{
            position: relative;
        }}

        .scatter-plot-wrap {{
            width: 100%;
            height: calc(100% - 28px);
        }}

        .scatter-plot-wrap .plotly-graph-div {{
            width: 100% !important;
            height: 100% !important;
        }}

        .scatter-x-axis-title {{
            position: absolute;
            left: 0;
            right: 0;
            bottom: 5px;
            height: 20px;
            text-align: center;
            color: white;
            font-family: Arial, sans-serif;
            font-size: 13px;
            line-height: 20px;
            pointer-events: none;
            z-index: 10;
        }}
    </style>
</head>

<body>
    <div class="dashboard-grid">
        <div class="panel map-panel">
            {map_html}
        </div>

        <div class="panel daily-panel">
            {daily_html}
        </div>

        <div class="panel scatter-panel scatter-panel-with-label">
            <div class="scatter-plot-wrap">
                {scatter_html}
            </div>

            <div class="scatter-x-axis-title">
                Annualized unique CAD events per 1,000 residents, log scale
            </div>
        </div>
    </div>

    <script>
        function resizePlots() {{
            if (window.Plotly) {{
                document.querySelectorAll(".js-plotly-plot").forEach(function(plot) {{
                    window.Plotly.Plots.resize(plot);
                }});
            }}
        }}

        window.addEventListener("load", function() {{
            resizePlots();
            setTimeout(resizePlots, 250);
            setTimeout(resizePlots, 1000);
            setTimeout(resizePlots, 2000);
        }});

        window.addEventListener("resize", resizePlots);
    </script>
</body>
</html>
"""

dashboard_path.write_text(full_dashboard_html, encoding="utf-8")

print(f"Saved dashboard to: {dashboard_path}")

webbrowser.open(dashboard_path.resolve().as_uri())